# 🧭 Chapter 05-01a: What Is RAG? Retrieval-Augmented Generation

---

## 📖 The Story: The Expert With a Library Card

Imagine you hire an expert consultant. They're brilliant — but they graduated in 2023. Ask them about your company's Q4 2024 report, and they'll confidently make something up.

Now give that expert a **library card** — access to your company's documents. Before answering, they look up the relevant info, then combine their expertise with the facts.

That's **RAG**: Give the LLM access to your data before it answers.

---

## 🎯 What Is RAG?

```
╔══════════════════════════════════════════════════════════════════════╗
║  RAG — RETRIEVAL-AUGMENTED GENERATION                               ║
║                                                                      ║
║  A technique that enhances LLM responses by first retrieving        ║
║  relevant information from external sources, then using it as       ║
║  context for the generation step.                                   ║
║                                                                      ║
║  User Question → RETRIEVE relevant docs → AUGMENT prompt → LLM     ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## ❓ Why Do We Need RAG?

LLMs have fundamental limitations:

| Problem | Without RAG | With RAG |
|---------|------------|----------|
| **Knowledge cutoff** | "I don't know about events after 2023" | Retrieves current data |
| **Hallucination** | Makes up plausible-sounding lies | Grounds answers in real docs |
| **Private data** | Can't access your company docs | Searches your knowledge base |
| **Source attribution** | "I think..." | "According to [doc], page 4..." |
| **Cost** | Fine-tuning is expensive | Just add documents! |

```
╔══════════════════════════════════════════════════════════╗
║  RAG vs FINE-TUNING                                     ║
║                                                          ║
║  Fine-tuning = Teaching the model new skills/style      ║
║  RAG = Giving the model a reference book during exam    ║
║                                                          ║
║  Most production apps use RAG first. Fine-tune later.   ║
╚══════════════════════════════════════════════════════════╝
```

---

## 🏗️ The RAG Architecture

```
┌──────────────────────────────────────────────────────────────┐
│           THE RAG PIPELINE (6 STEPS)                         │
│                                                              │
│  📄 LOAD                                                     │
│  │  PDFs, websites, CSVs, databases, APIs                   │
│  ▼                                                           │
│  ✂️ SPLIT                                                    │
│  │  Break large docs into chunks (500-1000 tokens)          │
│  ▼                                                           │
│  🔢 EMBED                                                    │
│  │  Convert chunks to vectors (OpenAI, etc.)                │
│  ▼                                                           │
│  🗄️ STORE                                                    │
│  │  Save in vector DB (ChromaDB, Pinecone, etc.)            │
│  ▼                                                           │
│  🔍 RETRIEVE                                                 │
│  │  Find top-K relevant chunks for the user's question      │
│  ▼                                                           │
│  🤖 GENERATE                                                 │
│     LLM answers using retrieved context + question          │
└──────────────────────────────────────────────────────────────┘
```

Steps 1-4 happen **offline** (indexing phase).
Steps 5-6 happen **online** (query phase).

---

## 📄 Step 1: Document Loading

LangChain provides 100+ document loaders:

| Loader | Source | Package |
|--------|--------|--------|
| PyPDFLoader | PDF files | langchain-community |
| WebBaseLoader | Web pages | langchain-community |
| CSVLoader | CSV files | langchain-community |
| NotionDBLoader | Notion pages | langchain-community |
| DirectoryLoader | Entire folders | langchain-community |
| UnstructuredLoader | Any document | unstructured |

Each loader returns `Document` objects with:
- `page_content`: The text
- `metadata`: Source info (filename, page number, etc.)

---

## ✂️ Step 2: Text Splitting

Why split? LLMs have context windows. A 200-page PDF won't fit.

```
╔══════════════════════════════════════════════════════════╗
║  TEXT SPLITTING STRATEGIES                               ║
║                                                          ║
║  RecursiveCharacterTextSplitter (most common)           ║
║  ├── Splits by: \n\n → \n → space → character           ║
║  ├── Respects natural boundaries                         ║
║  └── chunk_size=1000, chunk_overlap=200                  ║
║                                                          ║
║  SemanticChunker (advanced)                              ║
║  ├── Splits where meaning changes                        ║
║  └── Uses embeddings to detect topic shifts              ║
║                                                          ║
║  TokenTextSplitter (precise)                             ║
║  ├── Splits by token count (not characters)              ║
║  └── More accurate for LLM context limits                ║
╚══════════════════════════════════════════════════════════╝
```

**The overlap trick**: Adjacent chunks share some text, so context isn't lost at boundaries.

```
Chunk 1: [........text........]
Chunk 2:            [........text........]
                    ^^^^^ overlap ^^^^^
```

---

## 🔢 Steps 3-4: Embed & Store

We covered this in Chapter 4!

```
Chunks → OpenAI Embeddings → ChromaDB / Pinecone / FAISS
```

---

## 🔍 Step 5: Retrieval

When a user asks a question:

```
User: "What was our Q4 revenue?"
        │
        ▼ embed
[0.2, -0.4, 0.8, ...]
        │
        ▼ similarity search
Top 3 chunks:
1. "Q4 2024 revenue reached $45M, up 23%..." (score: 0.92)
2. "Revenue breakdown by product line..."      (score: 0.87)
3. "Compared to Q3, the growth rate..."        (score: 0.84)
```

LangChain retrievers:
- `vectorstore.as_retriever(search_kwargs={"k": 4})`
- `MultiQueryRetriever` — generates multiple search queries
- `SelfQueryRetriever` — extracts metadata filters from natural language
- `EnsembleRetriever` — combines vector + keyword search

---

## 🤖 Step 6: Generation

The LLM receives a prompt like:

```
Use the following context to answer the question.
If you don't know, say "I don't know."

Context:
{retrieved_chunks}

Question: {user_question}

Answer:
```

The key instruction: **"If you don't know, say I don't know"** — prevents hallucination!

---

## 🎓 Interview Corner

**Q: What are the common failure modes of RAG?**

<details>
<summary>💡 Click for Expert Answer</summary>

1. **Bad retrieval**: Wrong chunks returned → wrong answer. Fix: better embeddings, hybrid search, re-ranking
2. **Bad chunking**: Context split across chunks → incomplete info. Fix: overlap, semantic chunking
3. **Hallucination despite context**: LLM ignores context. Fix: stronger system prompt, lower temperature
4. **Context window overflow**: Too many chunks. Fix: limit k, summarize chunks, use map-reduce
5. **Stale data**: Embeddings not updated. Fix: incremental indexing pipeline
6. **Query-document mismatch**: Questions phrased differently than docs. Fix: HyDE, query rewriting

</details>

**Q: RAG vs Fine-tuning — when do you use which?**

<details>
<summary>💡 Click for Expert Answer</summary>

| Use Case | RAG | Fine-tuning |
|----------|-----|------------|
| Knowledge that changes often | ✅ | ❌ |
| Need citations/sources | ✅ | ❌ |
| Custom writing style | ❌ | ✅ |
| Domain-specific reasoning | ❌ | ✅ |
| Both? | ✅ RAG + fine-tuned model |

Start with RAG. Add fine-tuning if RAG alone isn't enough.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 RAG = Retrieve relevant docs → Augment prompt → Generate.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 6 steps: Load → Split → Embed → Store → Retrieve → Generate. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Chunking strategy matters more than most people think.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 RAG is cheaper and more flexible than fine-tuning.             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 "If you don't know, say I don't know" = anti-hallucination.   │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On — Chat with Your PDF in 50 Lines!

We'll build a complete RAG pipeline from scratch.

---

### 🎉 You Understand RAG!
The #1 most-asked topic in GenAI interviews. Let's build it! 🚀